# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os
import pandas as pd

# Clone the repository into this Colab runtime
if not os.path.exists("/content/flyrank-ml-starter"):
    !git clone https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git /content/flyrank-ml-starter

# Load the starter dataset
data_path = "/content/flyrank-ml-starter/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Loaded from:", data_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

Loaded from: /content/flyrank-ml-starter/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [16]:
# ### Ranked action queue

# The model output is used as a prioritization signal rather than an automatic refresh decision. Pages are ranked by their predicted probability of belonging to the observed `down` trend category.

# Each recommended page receives a reason code based on observable content and performance signals. These reason codes make the ranking easier for a human reviewer to interpret.

# ### Reason codes

# - `DECLINE_RISK_HIGH` — high model score indicating strong association with the observed `down` label.
# - `RECENT_PERFORMANCE_WEAK` — recent 30-day impressions or clicks are relatively low compared with the page's broader history.
# - `CONTENT_STALE` — the page has a high number of days since its last update.
# - `LOW_VISIBILITY` — the page has relatively weak search visibility based on available impression or position signals.
# - `ENGAGEMENT_REVIEW` — available engagement signals suggest the page may warrant further review.

# The final action is not determined by the model alone. A human reviewer should inspect the page and its context before deciding whether to refresh, monitor, merge, or take no action.

In [17]:
from sklearn.model_selection import GroupShuffleSplit

# Target: observed "down" trend
y = df["trend_direction"].eq("down").astype(int)

# Remove target, leakage-risk column, and grouping column
X = df.drop(
    columns=["trend_direction", "trend_pct", "client_id"]
)

# Client groups are used only for splitting
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify numeric and categorical features
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Numeric preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Logistic regression model
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

# Train
model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [19]:
# Generate predicted probability of the observed "down" label

y_prob = model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(y_prob))
print("Minimum probability:", round(y_prob.min(), 4))
print("Maximum probability:", round(y_prob.max(), 4))
print("Mean probability:", round(y_prob.mean(), 4))

Number of test predictions: 6163
Minimum probability: 0.0
Maximum probability: 1.0
Mean probability: 0.5187


In [20]:
# Build ranked action queue

action_queue = X_test.copy()

# Add model score and observed label
action_queue["predicted_probability"] = y_prob
action_queue["observed_trend"] = df.iloc[test_idx]["trend_direction"].values

# Rank pages from highest to lowest model score
action_queue = action_queue.sort_values(
    "predicted_probability",
    ascending=False
).reset_index(drop=True)

action_queue["priority_rank"] = action_queue.index + 1

# Create human-readable reason codes
def assign_reason(row):
    reasons = []

    if row["predicted_probability"] >= 0.80:
        reasons.append("DECLINE_RISK_HIGH")

    if row["days_since_last_update"] >= df["days_since_last_update"].median():
        reasons.append("CONTENT_STALE")

    if row["impressions_last_30d"] < row["impressions_90d"] / 3:
        reasons.append("RECENT_PERFORMANCE_WEAK")

    if row["avg_position"] > df["avg_position"].median():
        reasons.append("LOW_VISIBILITY")

    if row["engagement_rate"] < df["engagement_rate"].median():
        reasons.append("ENGAGEMENT_REVIEW")

    if not reasons:
        reasons.append("REVIEW")

    return ", ".join(reasons)

action_queue["reason_codes"] = action_queue.apply(
    assign_reason,
    axis=1
)

# Display top 20
display(
    action_queue[
        [
            "priority_rank",
            "content_id",
            "predicted_probability",
            "reason_codes",
            "observed_trend"
        ]
    ].head(20)
)

,priority_rank,content_id,predicted_probability,reason_codes,observed_trend
0,1,content_ca17a024f90c,1.0,"DECLINE_RISK_HIGH, RECENT_PERFORMANCE_WEAK",down
1,2,content_007d1f134801,1.0,"DECLINE_RISK_HIGH, CONTENT_STALE, RECENT_PERFO...",down
2,3,content_b138c0b35a91,1.0,"DECLINE_RISK_HIGH, RECENT_PERFORMANCE_WEAK",down
3,4,content_05b68f8f80d4,1.0,"DECLINE_RISK_HIGH, CONTENT_STALE, RECENT_PERFO...",down
4,5,content_4092ad6d1f71,1.0,"DECLINE_RISK_HIGH, RECENT_PERFORMANCE_WEAK",down
5,6,content_d7175187ff12,1.0,"DECLINE_RISK_HIGH, RECENT_PERFORMANCE_WEAK",down
6,7,content_60a90d0ba16a,1.0,"DECLINE_RISK_HIGH, CONTENT_STALE, RECENT_PERFO...",down
7,8,content_5fe46e04994d,1.0,"DECLINE_RISK_HIGH, CONTENT_STALE, RECENT_PERFO...",down
8,9,content_c9147cda01f0,1.0,"DECLINE_RISK_HIGH, CONTENT_STALE, RECENT_PERFO...",down
9,10,content_10a4180682c1,1.0,"DECLINE_RISK_HIGH, CONTENT_STALE, RECENT_PERFO...",down


In [21]:
# ### Observed result

# The ranked queue places pages with the highest predicted probability of the observed `down` trend at the top. In the grouped test set, the first 20 ranked pages were all associated with the observed `down` label, consistent with the measured Precision@20 of 1.000.

# The reason codes provide additional context for human review, including high decline-risk score, content staleness, recent performance weakness, low visibility, and engagement review.

# The queue is a prioritization tool. It does not automatically determine that a page should be refreshed.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [22]:
# ### Intended use

# The action playbook is intended for SEO and content teams as a decision-support tool. It can help prioritize which webpages deserve review first by ranking pages associated with the observed `down` trend and providing simple reason codes.

# The output is intended to support human review and resource prioritization. It is not intended to automatically publish, rewrite, delete, merge, or refresh content.

# ### Limits

# The ranking is based on historical observations in the available dataset. The observed `down` label is a proxy and does not represent a measured outcome after a content refresh.

# The model therefore cannot establish that refreshing a ranked page will improve search performance. It also cannot establish or predict Google's ranking algorithm.

# Recommendations should be reviewed using additional context that is not represented in the model before any content action is taken.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.